## Use this Jupyter Notebook to recreate the numerical results, as well as the corresponding plots and figures and from the publication
## [On the Use of Bagging for Local Intrinsic Dimensionality Estimation](https://linktopaper).

We start with a few imports.

In [1]:
#Import garbage collector for handling multiple experiments: loading data -> creating plot -> pick up data with garbage collector -> repeat
import gc

We fix the root directory to the project folder, in case it was improperly set up before for some reason.

In [7]:
#Setup project directory path
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)
p = str(PROJECT_ROOT)
if p not in sys.path:
    sys.path.insert(0, p)

In [2]:
#Import internal functions and presets
from Bagging_for_LID.run_files.final_tasks import *

We set up the variables for computing/loading the results:

- **load** (`bool`): If `true`, we load already performed (downloaded or saved) experiment objects as .pkl files, from the `pkl_directory` folder.
- **load_data** (`bool`): If `true`, we load already generated datasets as .pkl files, from the `pkl_directory` folder. If `false` every experiment object will generate a new dataset. Note that this variable has an effect on the results only if `load = False`.
- **pkl_directory** (`path`): Path to directory where we save and load datasets and completed experiment objects from as .pkl files.
- **multiprocess** (`bool`): If `true`, we allow the computation across the range of experiments to be paralellized using multiple cores.
- **worker_count** (`int`, default =`None`): If `multiprocess = true`, this determines the number of cores used. The default case means using as many cores as possible.
- **save_name** (`str`): Prefix for the name of all saved experiment objects and plots.

In [8]:
#Setup computation
load=True #Load already complete experiment .pkl files.
load_data=True #Load data for running experiments, if load=True, this has no effect on result output.
pkl_directory = r'C:\pkls' #Directory for saving and loading
multiprocess=True #Toggle multiprocessing
worker_count=7 #Worker count for multiprocessing.
save_name='mergedresult' #Save and load .pkl files with this name prefix. Amongst the downloadable files at Zenodo, the larger files with 'mergedresult' contain all data required to obtain the results (including datasets). While the 'light_mergedresult' smaller files have only the data necessary to reconstruct the plots, they are more like data storage, not interactive class objects.
plot_style = 'main' #Use the 'main' or 'appendix' presets. Sets the style of plots as they are in the main paper/in the appendix.
#RESULTS ARE SAVED AT the ./Output folder.
#Modify output folders at Bagging_for_LID.run_files.final_tasks

We run a set of experiments simply to generate and save the datasets at `pkl_directory`, for later loading. This step is only necessary if `load=False` and we want each experiment on the same data distribution to have the exact same sample set.

In [9]:
#Run or load a mock experiment to generates all datasets (to be loaded and reused in different experiments, if we want to recreate them)
if not load:
    results_data = new_result_generator(param_dicts_data, multiprocess=False, load=False, load_data=False,
                                        worker_count=None,
                                        save_name='data_generation',
                                        directory=pkl_directory)

<_io.BufferedReader name='C:\\pkls\\data_generation'>


We create the list of tasks to perform. There are five presets, corresponding to the experimental results in the publication.

- **Bagging and Smoothing test**: Evaluates the `baseline`, `baseline with smoothing`, `simple bagging`, `simple bagging with pre-smoothing`, `simple bagging with post smoothing`, `simple bagging with pre-smoothing and post-smoothing` methods, across a range of k-NN hyperparameters (k) and sampling rate (sr) hyperparameters for bagging.
- **Number of bags test**: MSE decomposition bar charts for each dataset as a function of the number of bags used for the bagged LID estimator.
- **Sampling rate test**: MSE decomposition bar charts for each dataset as a function of the sampling rate used for the bagged LID estimator.
- **Interaction of sampling rate (sr) and number of bags (B)**: Heatmaps of the relative difference between the MSE achieved by the baseline estimator and the MSE of its bagged counterpart for each dataset and cross-combinations of values for the sampling rate (x-axis) and the number of bags (y-axis).
- **Interaction of k-NN hyperparameter (k) and sampling rate (sr)**: Heatmaps of the relative difference between the MSE achieved by the baseline estimator and the MSE of its bagged counterpart for each dataset and cross-combinations of values for the sampling rate (x-axis) and the k-NN neighborhood size (y-axis).

Leave out any key-value pair from `task_dict` in order to ignore running that experiment. Then run the cell.

In [7]:
#Setup result generation
#Bagging and smoothing test (baseline, baseline with smoothing, simple bagging, simple bagging with pre or/and post smoothing)
#Number of bags test (mse bar charts)
#Sampling rate test (mse bar charts)
#Interaction of sampling rate and number of bags (mse difference heatmaps)
#Interaction of k and sampling rate (mse difference heatmaps)

task_dict = {"Bagging_and_smoothing_test": effectiveness_test_general_param_dict,
             "Number_of_bags_test": Nbag_test_general_param_dict,
             "Sampling_rate_test": sr_test_general_param_dict,
             "Interaction_of_sampling_rate_and_number_of_bags_test": interaction_sr_Nbag_test_general_param_dict,
             "Interaction_of_k_and_sampling_rate_test": interaction_sr_k_test_general_param_dict}
tasks = setup_tasks(task_dict, multiprocess=multiprocess, load=load, load_data=load_data, worker_count=worker_count, save_name=save_name, directory=pkl_directory)
plot_tasks = plot_tasks_main if plot_style == 'main' else plot_tasks_appendix

Run the following cell to create plots from loaded data, or to regenerate the data and create plots afterwards. Note that regenerating every experiment can take tens of hours.

Plots are saved at the `./Output` folder inside the Bagging_for_LID project directory. Change this at any time at the beginning of the Bagging_for_LID.run_files.final_tasks file.

In [ ]:
#Run results and plotting at the same time to save RAM
#calculating/loading data -> creating plot -> pick up data with garbage collector -> repeat
for key, func, args, kwargs in tasks:
    ok, result_dict = run_task_safely(func, *args, **kwargs)
    if not ok:
        continue
    try:
        consume_and_plot(key, result_dict, plot_tasks)
    finally:
        del result_dict
        gc.collect()